<a href="https://colab.research.google.com/github/Ilhamlafeer/Airline_Sentiment_Analysis_Using_BERT/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pandas scikit-learn datasets transformers torch accelerate streamlit

In [3]:
!pip uninstall -y torchvision

In [4]:
import os
import json
import numpy as np
import pandas as pd

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [5]:
# CONFIGURATION

DATA_PATH = "/content/drive/MyDrive/NLP/Tweets.csv"
MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "/content/drive/MyDrive/NLP/airline_sentiment_bert"

MAX_LENGTH = 128
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [6]:
# LOAD DATA

print("Loading dataset...")

df = pd.read_csv(DATA_PATH)

# Keep only required columns
df = df[["text", "airline_sentiment"]]

# Remove missing values
df = df.dropna()

# Remove duplicate tweets
df = df.drop_duplicates()

print(f"Dataset size: {len(df)}")

print("\nSentiment distribution:")
print(df["airline_sentiment"].value_counts())

Loading dataset...
Dataset size: 14452

Sentiment distribution:
airline_sentiment
negative    9087
neutral     3067
positive    2298
Name: count, dtype: int64


In [7]:
# ENCODE LABELS
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["airline_sentiment"]
)

print("\nLabel mapping:")

for index, label in enumerate(label_encoder.classes_):
    print(f"{index} -> {label}")


# Save label mapping
os.makedirs(OUTPUT_DIR, exist_ok=True)

label_mapping = {
    str(index): label
    for index, label in enumerate(label_encoder.classes_)
}

with open(
    os.path.join(OUTPUT_DIR, "label_mapping.json"),
    "w"
) as f:
    json.dump(label_mapping, f, indent=4)


Label mapping:
0 -> negative
1 -> neutral
2 -> positive


In [8]:
# 4. TRAIN / TEST SPLIT
train_df, test_df = train_test_split(
    df[["text", "label"]],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

print("\nTraining samples:", len(train_df))
print("Testing samples:", len(test_df))


# Convert Pandas to Hugging Face Dataset

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


Training samples: 11561
Testing samples: 2891


In [9]:
# LOAD BERT TOKENIZER
print("\nLoading tokenizer...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)


Loading tokenizer...


In [10]:
# TOKENIZATION
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )


print("Tokenizing dataset...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


# Remove text column
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])


# Set PyTorch format
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Tokenizing dataset...


Map:   0%|          | 0/11561 [00:00<?, ? examples/s]

Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

In [11]:
# 7. LOAD PRETRAINED BERT
print("\nLoading BERT model...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)



Loading BERT model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# METRICS

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [13]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    logging_steps=50,

    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,

    seed=RANDOM_STATE,

    fp16=True,

    report_to="none"
)

In [14]:
# TRAINER

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)


In [15]:
# TRAIN

print("\nStarting BERT training...")

trainer.train()



Starting BERT training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.413394,0.440927,0.845728,0.842528,0.845728,0.842935
2,0.287833,0.528309,0.848495,0.848196,0.848495,0.847279


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.413394,0.440927,0.845728,0.842528,0.845728,0.842935
2,0.287833,0.528309,0.848495,0.848196,0.848495,0.847279
3,0.223872,0.680132,0.855067,0.853295,0.855067,0.853999


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=4338, training_loss=0.3332332361123704, metrics={'train_runtime': 384.2209, 'train_samples_per_second': 90.268, 'train_steps_per_second': 11.29, 'total_flos': 2281390666765056.0, 'train_loss': 0.3332332361123704, 'epoch': 3.0})

In [16]:
# EVALUATION

print("\nEvaluating model...")

results = trainer.evaluate()

print("\nEvaluation Results:")

for key, value in results.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

    else:
        print(f"{key}: {value}")


Evaluating model...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.223872,0.680132,3,0.855067,0.853295,0.855067,0.853999



Evaluation Results:
eval_loss: 0.6801
eval_accuracy: 0.8551
eval_precision: 0.8533
eval_recall: 0.8551
eval_f1: 0.8540


In [17]:
# DETAILED CLASSIFICATION REPORT

print("\nGenerating classification report...")

predictions = trainer.predict(test_dataset)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=label_encoder.classes_
    )
)



Generating classification report...


              precision    recall  f1-score   support

    negative       0.91      0.92      0.92      1818
     neutral       0.74      0.69      0.71       613
    positive       0.79      0.80      0.80       460

    accuracy                           0.86      2891
   macro avg       0.81      0.81      0.81      2891
weighted avg       0.85      0.86      0.85      2891



In [18]:
# CONFUSION MATRIX
cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[1677   97   44]
 [ 133  425   55]
 [  35   55  370]]


In [19]:
# SAVE MODEL
print("\nSaving model...")

model.save_pretrained(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print("\nModel saved to:")
print(OUTPUT_DIR)

print("\nTraining completed successfully!")


Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/drive/MyDrive/NLP/airline_sentiment_bert

Training completed successfully!
